# 02 - Explore Dataset & Preliminary Cleaning

We are now going to look at the merged dataset and get an intuition of what's happening.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

dft = pd.read_csv('../data/STATS19/dft_STATS19_1979_23_SY.csv', low_memory=False)

In [2]:
# dft.info()

In [5]:
def nulls_and_minus1_summary(df):
    summary = pd.DataFrame({
        'Column': df.columns,
        'Data Type': df.dtypes.values,
        'Null Count': df.isnull().sum().values,
        '% Missing': (df.isnull().sum() / len(df) * 100).round(2),
        '% -1 Values': [
            ((df[col] == -1) | (df[col] == -1.0)).sum() / len(df) * 100 
            if pd.api.types.is_numeric_dtype(df[col]) else None
            for col in df.columns
        ]
    })
    return summary.sort_values(by='Null Count', ascending=False).reset_index(drop=True)

# # Generate and display the table
# null_minus1_summary = nulls_and_minus1_summary(dft)
# with pd.option_context('display.max_rows', None):
#     display(null_minus1_summary)

We can highlight following columns that have some missing values:
- location_easting_osgr

In [6]:
columns_to_drop = [
  'dir_from_e',
  'dir_from_n',
  'dir_to_e',
  'dir_to_n',
  'latitude',
  'longitude',
  'location_easting_osgr',
  'location_northing_osgr',
  'did_police_officer_attend_scene_of_accident',
  'enhanced_casualty_severity',
  'casualty_distance_banding',
  'driver_distance_banding',
  'pedestrian_road_maintenance_worker',
  'casualty_imd_decile',
  'vehicle_left_hand_drive'
]
# dft_filtered = dft.drop(columns=columns_to_drop)
# dft_filtered.info()


### Missing Value Analysis
Now that we have explored the dataset a bit, we can see some columns have missing data, especially the older data. So, we will do some analysis of the missing values, as follows:
1. Check for null values in all columns.
2. Visualise the missing values by year.

After the analysis, we will replace '-1' values with NaN, so the dataset is more compatible with Pandas.

In [7]:
def plot_missing_values_by_year(df, year_col='accident_year'):
    # Step 1: Select columns with at least one null
    missing_cols = df.columns[df.isnull().any()].tolist()

    # Step 2: Group by year and count missing values for each column
    missing_by_year = df.groupby(year_col)[missing_cols].apply(lambda x: x.isnull().sum())

    # Step 3: Plot each column in a subplot
    num_cols = len(missing_cols)
    cols = 3
    rows = (num_cols + cols - 1) // cols  # auto-determine rows based on cols

    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), sharex=True)
    axes = axes.flatten()

    for i, col in enumerate(missing_cols):
        axes[i].plot(missing_by_year.index, missing_by_year[col], marker='o')
        axes[i].set_title(f"{col} - Missing Count")
        axes[i].set_xlabel("Year")
        axes[i].set_ylabel("Missing")
        axes[i].grid(True)

    # Remove any unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

# 🔧 Call the function on your dataset
# plot_missing_values_by_year(dft)


### Checking missing values for each parameter over the years
We will use an interactive tool that lets us check one parameter at a time and explore how missing values change over the years.

By selecting a column from the dropdown, we can see the percentage of missing values (including both NaN and -1) across different years. This helps us:
* Understand when certain features start being recorded properly
* Spot sudden changes in data quality or reporting
* Decide which years to include in our analysis based on data completeness

In [4]:
import sys
sys.path.append('../utilities')

from interactive_column_plotter import advanced_missing_explorer
import pandas as pd

In [5]:
# dft = pd.read_csv('data\STATS19\dft_STATS19_1979_23_SY.csv', low_memory=False)
advanced_missing_explorer(dft)


In [9]:
# for i in dft.columns:
#   print(i)
dft.shape

(243191, 85)